# M3L4 E01 — MiniTracer en Python puro
### Módulo 3 · Lecture 4 · Construcción, pruebas y trazabilidad de agentes en producción

---

## Qué necesitas saber antes

| Módulo | Concepto | Por qué lo necesitás acá |
|---|---|---|
| M3L4 E00 | Trace, Span, jerarquía | MiniTracer implementa exactamente esa estructura en Python |
| M3L1 | Tool contracts | Un span registra input/output como lo hace una tool contract |
| M3L2 | Agentes con estado | El tracer mantiene estado interno (`self.traces`) |
| Python | `time.time()`, `uuid.uuid4()`, `datetime` | Los vamos a usar como herramientas base |

Si no viste E00, empezá por ahí. Este ejercicio construye el tracer que E00 solo describe.

---

## Definiciones clave

| Concepto | Definición simple | Cómo aparece en este notebook |
|---|---|---|
| **MiniTracer** | Clase Python que implementa tracing en memoria | `class MiniTracer` con métodos `start_trace`, `add_span`, etc. |
| **trace_id** | Identificador único por request (UUID) | `str(uuid.uuid4())` |
| **span_id** | Identificador único por paso dentro de un trace | `str(uuid.uuid4())` dentro de cada span |
| **start_trace()** | Método que crea un nuevo trace con estado inicial | Devuelve un dict con trace_id, spans vacío, timestamp |
| **add_span()** | Método que agrega un paso (span) a un trace existente | Agrega un dict a `trace['spans']` |
| **update_trace_output()** | Método que completa el trace con la respuesta final | Asigna `trace['output']` y calcula `total_duration_ms` |
| **show_trace()** | Método que devuelve el trace para inspección | `return trace` |
| **Generation** | Sub-tipo de span que representa una llamada a un LLM | No se implementa acá, pero es el siguiente nivel jerárquico |

---

## Cómo encaja esto en un sistema de agentes

```
E00: Concepto de Trace y Span (teoría)
    |
    v
E01: MiniTracer en Python puro (implementación)
    |  Trace -> Span -> estructura en diccionarios
    v
E02: MiniTracer aplicado a sistema multi-agente
    |  Routing + agente especialista como spans
    v
E03: Diagnóstico de fallas leyendo traces
    |  Misclassification, retrieval vacío, latencia, loops
    v
E04+: Langfuse (tracing profesional, visual y persistente)
```

**Objetivo del ejercicio:** construir un sistema de tracing propio para entender cómo funciona Langfuse desde adentro.

## Instalación e imports

Este notebook solo usa la biblioteca estándar de Python. Cada import tiene una función específica:

| Import | Qué hace | Por qué lo necesitamos |
|---|---|---|
| `import time` | Acceso a `time.time()` para medir duración en ms | Para calcular `duration_ms` de spans y `total_duration_ms` del trace |
| `import uuid` | Genera identificadores únicos con `uuid.uuid4()` | Para crear `trace_id` y `span_id` únicos |
| `from datetime import datetime` | Timestamp ISO con `datetime.utcnow().isoformat()` | Para registrar cuándo se creó cada trace |
| `from typing import Optional, Any` | Tipado opcional para las firmas de métodos | Para documentar qué parámetros pueden ser `None` |

> **Nota:** `typing` es solo para documentación. Python no obliga a usarlo, pero ayuda a que el código sea auto-documentado.

```python
import time
import uuid
from datetime import datetime
from typing import Optional, Any
```

## Clase MiniTracer — Esqueleto

A continuación tenés la clase base. Tu tarea es **completar los TODO** para que funcione correctamente.

### Qué hace cada método

| Método | Recibe | Devuelve | Efecto secundario |
|---|---|---|---|
| `start_trace(name, input_data, metadata, tags)` | Nombre, input, metadata, tags | `dict` (el trace creado) | Lo agrega a `self.traces` |
| `add_span(trace, name, input_data, output_data, metadata, duration_ms)` | Un trace existente + datos del span | `dict` (el span creado) | Lo agrega a `trace['spans']` |
| `update_trace_output(trace, output_data)` | Un trace + output final | `None` | Modifica `trace['output']` in-place |
| `show_trace(trace)` | Un trace | `dict` | Solo retorna el trace |

### Estructura interna de un trace

```python
trace = {
    'trace_id': 'abc-123',          # único por request
    'name': 'support-request',       # nombre descriptivo
    'input': {'query': '...'},       # lo que recibió el sistema
    'output': {'answer': '...'},     # lo que devolvió (se actualiza al final)
    'metadata': {'environment': 'notebook'},  # contexto adicional
    'tags': ['demo', 'm3l4'],        # etiquetas para filtrar
    'spans': [                       # lista de pasos ejecutados
        {'span_id': 'def-456', 'name': 'routing', ...},
        {'span_id': 'ghi-789', 'name': 'agent', ...}
    ],
    'created_at': '2025-01-01T00:00:00',  # timestamp ISO
    '_started_at_ts': 1234567890.0   # timestamp epoch (interno, se usa para calcular duración)
}
```

> **Nota:** `_started_at_ts` empieza con `_` por convención Python: es un campo interno que no debería mostrarse al usuario final.

In [ ]:
import time
import uuid
from datetime import datetime
from typing import Optional, Any

In [ ]:
class MiniTracer:
    def __init__(self):
        self.traces = []

    def start_trace(self, name: str, input_data=None, metadata=None, tags=None) -> dict:
        """
        Crea y registra un nuevo trace.

        Args:
            name: nombre descriptivo del trace (ej: 'support-request')
            input_data: datos de entrada (query del usuario, etc.)
            metadata: diccionario con info adicional (environment, user_id, etc.)
            tags: lista de etiquetas para filtrar (ej: ['demo', 'm3l4'])

        Returns:
            dict con el trace creado
        """
        # TODO: crear el diccionario 'trace' con:
        # - trace_id: UUID único
        # - name: el nombre recibido
        # - input: input_data
        # - output: None (se completa al final)
        # - metadata: metadata or {}
        # - tags: tags or []
        # - spans: lista vacía
        # - created_at: timestamp ISO
        trace = {
            # TODO
        }
        self.traces.append(trace)
        return trace

    def add_span(self, trace: dict, name: str, input_data=None,
                 output_data=None, metadata=None, duration_ms=None) -> dict:
        """
        Agrega un span a un trace existente.

        Args:
            trace: el trace al cual agregar el span
            name: nombre del span (ej: 'orchestrator-routing', 'hr-agent')
            input_data: entrada del span
            output_data: salida del span
            metadata: info adicional del span
            duration_ms: duración en milisegundos

        Returns:
            dict con el span creado
        """
        span = {
            # TODO: agregar span_id, name, input, output, metadata, duration_ms
        }
        # TODO: agregar el span a trace['spans']
        return span

    def update_trace_output(self, trace: dict, output_data):
        """
        Actualiza el output del trace.

        Args:
            trace: el trace a actualizar
            output_data: respuesta final del sistema
        """
        # TODO: asignar trace['output'] = output_data
        pass

    def show_trace(self, trace: dict) -> dict:
        """
        Devuelve el trace para inspección.

        Args:
            trace: el trace a mostrar

        Returns:
            dict con toda la información del trace
        """
        return trace

print('MiniTracer listo.')

## Ejecución — Probá tu MiniTracer

Una vez completados los TODO, esta celda debe ejecutarse sin errores y mostrar el trace completo.

In [ ]:
tracer = MiniTracer()

trace = tracer.start_trace(
    name='support-request',
    input_data={'query': 'No puedo ver mi factura'},
    metadata={'environment': 'notebook', 'user_id': 'student-01'},
    tags=['demo', 'm3l4']
)

tracer.add_span(
    trace,
    name='orchestrator-routing',
    input_data={'query': 'No puedo ver mi factura'},
    output_data={'intent': 'finance'},
    duration_ms=120
)

tracer.add_span(
    trace,
    name='finance-agent',
    input_data={'query': 'No puedo ver mi factura'},
    output_data={'answer': 'Podés ver tu factura desde el portal de pagos.'},
    duration_ms=840
)

tracer.update_trace_output(
    trace,
    {'final_answer': 'Podés ver tu factura desde el portal de pagos.'}
)

tracer.show_trace(trace)

## TODO — Checks de validación

Completa las aserciones para verificar que tu MiniTracer funciona correctamente.

In [ ]:
# TODO: verificar que el trace tiene las claves correctas
assert 'trace_id' in trace,      'Falta trace_id'
assert 'name' in trace,          'Falta name'
assert 'spans' in trace,         'Falta spans'
assert 'input' in trace,         'Falta input'
assert 'output' in trace,        'Falta output'
assert 'metadata' in trace,      'Falta metadata'
assert 'tags' in trace,          'Falta tags'

# TODO: verificar que hay 2 spans
assert len(trace['spans']) == 2, f'Se esperaban 2 spans, hay {len(trace["spans"])}'

# TODO: verificar el output final
assert trace['output'] is not None, 'Output no fue actualizado'

# TODO: verificar tags
assert 'm3l4' in trace['tags'], 'Falta el tag m3l4'

print('Checks E01 OK')

## Extensión — Agregar timestamp de duración total

Modifica `start_trace` y `update_trace_output` para que al cerrar el trace calcule automáticamente la **duración total** en ms.

```python
# Pista: guarda un campo 'started_at_ts' con time.time() al crear
# y al cerrar calcula: round((time.time() - started_at_ts) * 1000, 2)
```

Por qué importa: la duración total permite medir performance del sistema completo, no solo de pasos individuales.

In [ ]:
# TODO: extender MiniTracer para calcular duration_ms total
# (opcional — para quienes terminaron antes)
pass

## Errores comunes

| Error | Causa | Cómo detectarlo |
|---|---|---|
| `trace_id` no se genera | Olvidar `str(uuid.uuid4())` en start_trace | El trace no tiene identificador único |
| Span no se agrega al trace | No hacer `trace['spans'].append(span)` | `len(trace['spans'])` sigue siendo 0 |
| `output` queda `None` | No llamar `update_trace_output()` | El trace parece incompleto |
| Olvidar `self.traces.append(trace)` | El trace se crea pero no se guarda en el tracer | `tracer.traces` está vacío |
| `duration_ms` no se muestra | No pasar el parámetro al crear el span | El span existe pero sin métrica de tiempo |

## Síntesis

### Qué construiste

| Componente | Descripción | Equivalente en producción (Langfuse) |
|---|---|---|
| `MiniTracer` | Clase que gestiona traces en memoria | `Langfuse()` cliente |
| `start_trace()` | Crea un nuevo trace | `trace = langfuse.trace(...)` |
| `add_span()` | Agrega un paso al trace | `trace.span(...)` |
| `trace_id` / `span_id` | Identificadores únicos | Generados automáticamente |
| `duration_ms` | Duración por paso | Calculado automáticamente |
| `metadata` / `tags` | Contexto y etiquetas | Campos equivalents en Langfuse |

### Relación con otros ejercicios

| Ejercicio | Conexión con E01 |
|---|---|
| **E02** | Aplicás MiniTracer a un sistema multi-agente con orquestador |
| **E03** | Usás traces para diagnosticar fallas (alimentás tu MiniTracer a `diagnose_trace()`) |
| **E04** | Langfuse reemplaza a MiniTracer con tracing profesional automático |